# Elektriciteitstarieven: vast (DATS24) vs dynamisch (EPEX)

Vergelijking van de totale elektriciteitskost voor twee scenario's:

| Scenario | Beschrijving |
|---|---|
| **A** | Geen batterij — vast DATS24-tarief vs dynamisch EPEX-tarief |
| **B** | Met geoptimaliseerde batterij — dynamisch EPEX-tarief (minimale kost) |

**Netkosten (Fluvius)** zijn identiek in beide scenario's.

> *Tariefwaarden zijn representatieve 2025-schattingen.*
> *Pas de configuratiecellen aan indien je over exacte contractwaarden beschikt.*

In [1]:
import sys
from pathlib import Path

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display
from scipy.optimize import linprog
from scipy.sparse import lil_matrix

from scripts.config import FINAL_DIR, INTERMEDIATE_DIR
from scripts import epex_kwartier
import warnings; warnings.filterwarnings('ignore')

## 1. Tariefconfiguratie

### DATS24 leveringsprijs (exclusief netkosten, exclusief BTW)

Bronnen: VREG prijzendatabank, DareToCompare.be (april 2026).
Pas de waarden aan op basis van je eigen contract.

In [2]:
# ── DATS24 vast tarief ──────────────────────────────────────────────────
DATS24 = {
    'dag_kwh':       0.1479,   # €/kWh leveringsprijs dag (7:00-22:00 weekdagen)
    'nacht_kwh':     0.1240,   # €/kWh leveringsprijs nacht
    'injectie_kwh':  0.0655,   # €/kWh injectievergoeding (VREG apr-2026)
    'maandkost':     2.24,     # €/maand vaste leveringskost
    'btw':           0.06,     # BTW-factor (6%)
}

# ── Dynamisch tarief (EPEX SPOT + opslag) ────────────────────────────────
DYNAMISCH = {
    'opslag_kwh':      0.0140,  # €/kWh vaste opslag boven EPEX-prijs
    'injectie_korting':0.0050,  # €/kWh korting op EPEX bij teruglevering
    'maandkost':       2.24,    # €/maand vaste leveringskost
    'btw':             0.06,
}

# ── Fluvius netkosten 2025 (Vilvoorde) ───────────────────────────────────
# Identiek voor vast én dynamisch tarief.
NETKOSTEN = {
    # Capaciteitstarief (maandpiek in kW, minimum 2.5 kW)
    'cap_eur_kw_jaar':  44.60,   # €/kW/jaar
    'cap_min_kw':        2.5,    # kW minimum
    # Distributienetkosten afname
    'dist_dag_kwh':     0.0295,  # €/kWh dag
    'dist_nacht_kwh':   0.0138,  # €/kWh nacht
    # Transmissie (flat)
    'trans_kwh':        0.0042,  # €/kWh
    # Injectie netkosten (afname-richting bij teruglevering)
    'inj_net_kwh':      0.0016,  # €/kWh
    # Vaste meterkost
    'meter_mnd':        3.75,    # €/maand
    # Heffingen & bijdragen (REG, CREG, DVO, Energiefonds)
    'heffingen_kwh':    0.0218,  # €/kWh variabel
    'heffingen_mnd':    1.85,    # €/maand vast
    'btw':              0.06,
}

# ── Batterijparameters ────────────────────────────────────────────────────
BAT = {
    'cap_kwh':   6.5,   # Bruikbare capaciteit (kWh)
    'max_kw':    3.0,   # Max laad-/ontlaadvermogen (kW)
    'eta_c':     0.95,  # Laadrendement
    'eta_d':     0.95,  # Ontlaadrendement
    'soc_min':   0.05,  # Minimum SOC (fractie)
    'soc_max':   0.95,  # Maximum SOC (fractie)
    'soc_start': 0.30,  # Beginstand per dag (fractie)
}

for naam, d in [('DATS24', DATS24), ('DYNAMISCH', DYNAMISCH), ('NETKOSTEN', NETKOSTEN), ('BATTERIJ', BAT)]:
    print(f'{naam}: {d}')

DATS24: {'dag_kwh': 0.1479, 'nacht_kwh': 0.124, 'injectie_kwh': 0.0655, 'maandkost': 2.24, 'btw': 0.06}
DYNAMISCH: {'opslag_kwh': 0.014, 'injectie_korting': 0.005, 'maandkost': 2.24, 'btw': 0.06}
NETKOSTEN: {'cap_eur_kw_jaar': 44.6, 'cap_min_kw': 2.5, 'dist_dag_kwh': 0.0295, 'dist_nacht_kwh': 0.0138, 'trans_kwh': 0.0042, 'inj_net_kwh': 0.0016, 'meter_mnd': 3.75, 'heffingen_kwh': 0.0218, 'heffingen_mnd': 1.85, 'btw': 0.06}
BATTERIJ: {'cap_kwh': 6.5, 'max_kw': 3.0, 'eta_c': 0.95, 'eta_d': 0.95, 'soc_min': 0.05, 'soc_max': 0.95, 'soc_start': 0.3}


## 2. Data laden

### 2a. Kwartierdata (Fluvius basis)

In [3]:
# Laad overall.csv
df = pd.read_csv(ROOT / 'data/Final/overall.csv', parse_dates=['kwartier'])

# Afgeleide kolommen
df['datum'] = pd.to_datetime(df['kwartier']).dt.date
df['kwartier_van_dag'] = (
    pd.to_datetime(df['kwartier']).dt.hour * 4
    + pd.to_datetime(df['kwartier']).dt.minute // 15
)

# Batterijflows per kwartier (iLumen uurwaarden /4)
df['bat_charge_q']    = df['bat_geladen_kwh'].fillna(0)  / 4
df['bat_discharge_q'] = df['bat_ontladen_kwh'].fillna(0) / 4

# Basisbelasting zonder batterij:
#   net_load = afname - injectie = huis + ev + bat_charge - bat_discharge - zon
#   base_load_no_bat = net_load - bat_charge + bat_discharge
df['base_load_kwh'] = (
    df['afname_kwh'] - df['injectie_kwh']
    - df['bat_charge_q'] + df['bat_discharge_q']
)

print(f'Kwartieren  : {len(df)}')
print(f'Periode     : {df["kwartier"].min().date()} -> {df["kwartier"].max().date()}')
print(f'Basis afname: {df["base_load_kwh"].clip(lower=0).sum():.1f} kWh')
print(f'Basis inject: {(-df["base_load_kwh"]).clip(lower=0).sum():.1f} kWh')

Kwartieren  : 50104
Periode     : 2024-11-01 -> 2026-04-06
Basis afname: 12187.2 kWh
Basis inject: 1036.9 kWh


### 2b. EPEX SPOT-kwartierdata laden

De kwartierwaarden worden geladen uit het lokale tussenresultaatbestand
`data/intermediate results/epex_kwartieren.csv`, aangemaakt door
`scripts/epex_kwartier.py` via een antiderivaat-preserverende kubische spline.

**Bijwerklogica:**
- Bestaat `epex_kwartieren.csv` **niet** → conversie wordt automatisch aangeroepen.
- Is `epex_be.csv` **recenter** dan `epex_kwartieren.csv` → conversie wordt herhaald.
- Anders → bestand wordt rechtstreeks ingeladen (geen netwerkverkeer nodig).

De uurdata (`epex_be.csv`) wordt opgehaald via de **ENTSO-E API** in
`notebooks/data_voorbereiding.ipynb`. Zorg dat de data­voorbereidingsnotebook
eerst volledig is uitgevoerd voordat je dit notebook opent.

In [4]:
from scripts import epex_kwartier
from scripts.epex_kwartier import KWARTIER_CACHE, EPEX_CACHE

# ── Bijwerkdetectie en laden ──────────────────────────────────────────────
# De converteer()-functie checkt automatisch of conversie nodig is:
#   1. Kwartierbestand ontbreekt -> altijd converteren
#   2. Uurbestand (epex_be.csv) nieuwer dan kwartierbestand -> converteren
#   3. Anders -> bestand is actueel, enkel inladen
if not KWARTIER_CACHE.exists():
    if not EPEX_CACHE.exists():
        print('FOUT: epex_be.csv niet gevonden.')
        print('Voer eerst notebooks/data_voorbereiding.ipynb uit (cel 0g).')
        df_epex_kw = None
    else:
        print('Kwartierbestand ontbreekt - conversie starten...')
        epex_kwartier.converteer()
        df_epex_kw = epex_kwartier.laad()
elif EPEX_CACHE.exists():
    t_uur = EPEX_CACHE.stat().st_mtime
    t_kw  = KWARTIER_CACHE.stat().st_mtime
    if t_uur > t_kw:
        print('Uurbestand is nieuwer dan kwartierbestand - herconversie...')
        epex_kwartier.converteer(force=True)
    df_epex_kw = epex_kwartier.laad()
else:
    df_epex_kw = epex_kwartier.laad()

if df_epex_kw is not None and not df_epex_kw.empty:
    print(f'EPEX kwartieren geladen: {len(df_epex_kw)} rijen')
    print(f'  Bereik : {df_epex_kw.index.min()} -> {df_epex_kw.index.max()}')
    print(f'  Gem.   : {df_epex_kw["price_eur_mwh"].mean():.2f} €/MWh')
    print(f'  Min.   : {df_epex_kw["price_eur_mwh"].min():.2f} €/MWh')
    print(f'  Max.   : {df_epex_kw["price_eur_mwh"].max():.2f} €/MWh')

EPEX kwartieren geladen: 69984 rijen
  Bereik : 2024-04-11 00:00:00+02:00 -> 2026-04-09 23:45:00+02:00
  Gem.   : 80.53 €/MWh
  Min.   : -513.45 €/MWh
  Max.   : 585.80 €/MWh


In [5]:
# -- Koppel kwartier-EPEX-prijzen aan kwartierdata ------------------------
# df_epex_kw heeft een tz-aware index (Europe/Brussels).
# df['kwartier'] is tz-naive. We koppelen via een tz-naive hulpindex.
if df_epex_kw is not None and not df_epex_kw.empty:
    # Maak de EPEX-index tz-naive voor de join
    epex_naive = df_epex_kw.copy()
    epex_naive.index = epex_naive.index.tz_localize(None)

    # Verwijder eventuele duplicaten (ontstaan bij winteruurovergang:
    # 02:00 Brussels komt twee keer voor; tz_localize geeft dan twee
    # identieke naive timestamps). Behoud de laatste waarde.
    if epex_naive.index.duplicated().any():
        n_dup = epex_naive.index.duplicated().sum()
        print(f'  {n_dup} dubbele tijdstempels verwijderd (DST-overgang).')
        epex_naive = epex_naive[~epex_naive.index.duplicated(keep='last')]

    # Directe kwartier-op-kwartier join
    df['epex_eur_mwh'] = df['kwartier'].map(epex_naive['price_eur_mwh'])

    aanwezig  = df['epex_eur_mwh'].notna().sum()
    ontbrekend = df['epex_eur_mwh'].isna().sum()
    print(f'Kwartieren met EPEX-prijs : {aanwezig} / {len(df)}')
    print(f'Kwartieren zonder EPEX    : {ontbrekend}')
    if ontbrekend > 0:
        print('  (ontbrekende prijzen worden op de mediaanprijs gesteld in de berekeningen)')
else:
    print('WAARSCHUWING: geen EPEX-kwartierdata beschikbaar.')
    print('Dynamisch-tariefberekening wordt overgeslagen.')
    df['epex_eur_mwh'] = np.nan


  16 dubbele tijdstempels verwijderd (DST-overgang).
Kwartieren met EPEX-prijs : 50100 / 50104
Kwartieren zonder EPEX    : 4
  (ontbrekende prijzen worden op de mediaanprijs gesteld in de berekeningen)


## 3. Kostenberekenisfuncties

### 3a. Fluvius netkosten (identiek voor alle scenario's)

De **capaciteitstarief** is gebaseerd op de **hoogste 15-minuten-gemiddelde**
netafname per maand (minimaal 2,5 kW).

In [6]:
def netkosten_kwartier(df_in: pd.DataFrame) -> pd.Series:
    """
    Bereken de variabele netkosten (excl. BTW) per kwartier.
    Formule: (dist + trans + heffingen) per kWh afname + injectie-netkosten.
    """
    afname   = df_in['afname_kwh']
    injectie = df_in['injectie_kwh']
    tarief   = df_in['tarief']
    dist = np.where(tarief == 'dag', NETKOSTEN['dist_dag_kwh'], NETKOSTEN['dist_nacht_kwh'])
    var = afname * (dist + NETKOSTEN['trans_kwh'] + NETKOSTEN['heffingen_kwh'])
    var -= injectie * NETKOSTEN['inj_net_kwh']
    return pd.Series(var, index=df_in.index)


def capaciteitstarief_maand(afname_kwh: pd.Series,
                            kwartier_ts: pd.Series) -> pd.DataFrame:
    """
    Bereken het capaciteitstarief per maand op basis van de maandpiek (kW).

    Piek = maximum 15-min-gemiddeld vermogen [kW] = max(afname_kwh / 0.25).
    Minimumpiek = 2.5 kW.
    Jaarkosten = gem. maandpiek * cap_eur_kw_jaar; maandkost = 1/12 hiervan.
    """
    s = pd.Series(afname_kwh.values / 0.25,  # kW
                  index=pd.to_datetime(kwartier_ts))
    maandpiek = s.resample('MS').max().clip(lower=NETKOSTEN['cap_min_kw'])
    maandkost = maandpiek * NETKOSTEN['cap_eur_kw_jaar'] / 12
    return pd.DataFrame({'piek_kw': maandpiek, 'cap_kost': maandkost})


def vaste_netkosten_maand(kwartier_ts: pd.Series) -> pd.DataFrame:
    """
    Vaste maandelijkse netkosten (meter + heffingen vast), excl. BTW.
    """
    maanden = pd.to_datetime(kwartier_ts).to_period('M').drop_duplicates().to_timestamp()
    vaste = NETKOSTEN['meter_mnd'] + NETKOSTEN['heffingen_mnd']
    return pd.DataFrame({'vaste_nk': vaste}, index=maanden)

## 4. Scenario A — Zonder batterij

De **basisbelasting** (zonder batterij) is de netto gridflow die er zou zijn
als de batterij niet aanwezig was:

```
base_load = afname - injectie - bat_discharge + bat_charge
```

Positief = netto afname, negatief = netto injectie (overproductie zon).

| Kolom | Beschrijving |
|---|---|
| `afname_nb` | Netafname zonder batterij (kWh/kwartier) |
| `injectie_nb` | Injectie zonder batterij (kWh/kwartier) |

In [7]:
# Basisscenario: afname/injectie zonder batterij
df['afname_nb']   = df['base_load_kwh'].clip(lower=0)
df['injectie_nb'] = (-df['base_load_kwh']).clip(lower=0)

# Hulpkolom: tarief op basis van kwartier-tijdstip (zelfde als Fluvius-tarief)
# (tarief 'dag' of 'nacht' blijft ongewijzigd)

# ── Energiekosten zonder batterij ────────────────────────────────────────
def energiekosten_vast(afname_kwh, injectie_kwh, tarief):
    """Energiekost (excl. netkosten, excl. BTW) per kwartier — vast DATS24-tarief."""
    prijs = np.where(tarief == 'dag', DATS24['dag_kwh'], DATS24['nacht_kwh'])
    return afname_kwh * prijs - injectie_kwh * DATS24['injectie_kwh']

def energiekosten_dynamisch(afname_kwh, injectie_kwh, epex_eur_mwh):
    """Energiekost (excl. netkosten, excl. BTW) per kwartier — dynamisch tarief."""
    prijs_kwh   = epex_eur_mwh / 1000 + DYNAMISCH['opslag_kwh']
    verkoop_kwh = epex_eur_mwh / 1000 - DYNAMISCH['injectie_korting']
    return afname_kwh * prijs_kwh - injectie_kwh * verkoop_kwh


# Kosten per kwartier — Scenario A (zonder batterij)
nb_mask = df['epex_eur_mwh'].notna()  # alleen kwartieren met EPEX-prijs

df['nk_kwartier'] = netkosten_kwartier(df.assign(
    afname_kwh=df['afname_nb'], injectie_kwh=df['injectie_nb']
))

df['ek_vast_nb'] = energiekosten_vast(
    df['afname_nb'], df['injectie_nb'], df['tarief']
)
df['ek_dyn_nb'] = np.where(
    nb_mask,
    energiekosten_dynamisch(
        df['afname_nb'], df['injectie_nb'], df['epex_eur_mwh'].fillna(0)
    ),
    np.nan,
)

print('Totalen scenario A (zonder batterij):')
print(f'  Energiekost vast       : {df["ek_vast_nb"].sum():>8.2f} €')
print(f'  Energiekost dynamisch  : {df["ek_dyn_nb"].sum():>8.2f} € (waar EPEX beschikbaar)')
print(f'  Variabele netkosten    : {df["nk_kwartier"].sum():>8.2f} €')

Totalen scenario A (zonder batterij):
  Energiekost vast       :  1545.72 €
  Energiekost dynamisch  :  1318.83 € (waar EPEX beschikbaar)
  Variabele netkosten    :   550.67 €


### 4a. Maandoverzicht — Scenario A

In [8]:
df['maand'] = pd.to_datetime(df['kwartier']).dt.to_period('M')

# Capaciteitstarief per maand (zonder batterij = piek van base_load)
cap_nb = capaciteitstarief_maand(df['afname_nb'], df['kwartier'])

maand_a = df.groupby('maand').agg(
    ek_vast  =('ek_vast_nb',  'sum'),
    ek_dyn   =('ek_dyn_nb',   'sum'),
    nk_var   =('nk_kwartier', 'sum'),
    afname   =('afname_nb',   'sum'),
    injectie =('injectie_nb', 'sum'),
).round(2)

# Voeg capaciteitstarief en vaste kosten toe
if not isinstance(cap_nb.index, pd.PeriodIndex):
    cap_nb.index = cap_nb.index.to_period('M')
maand_a = maand_a.join(cap_nb[['piek_kw','cap_kost']], how='left')
maand_a['cap_kost'] = maand_a['cap_kost'].fillna(0)
maand_a['vast_mnd']  = NETKOSTEN['meter_mnd'] + NETKOSTEN['heffingen_mnd']
maand_a['lev_mnd']   = DATS24['maandkost']

btw = 1 + NETKOSTEN['btw']
maand_a['totaal_vast'] = (
    (maand_a['ek_vast'] + maand_a['lev_mnd']) * (1 + DATS24['btw'])
    + (maand_a['nk_var'] + maand_a['cap_kost'] + maand_a['vast_mnd']) * btw
)
maand_a['totaal_dyn'] = (
    (maand_a['ek_dyn'] + DYNAMISCH['maandkost']) * (1 + DYNAMISCH['btw'])
    + (maand_a['nk_var'] + maand_a['cap_kost'] + maand_a['vast_mnd']) * btw
)
maand_a['verschil'] = maand_a['totaal_dyn'] - maand_a['totaal_vast']
maand_a['verschil_pct'] = (maand_a['verschil'] / maand_a['totaal_vast'] * 100).round(1)

print('\nMAANDOVERZICHT — SCENARIO A (zonder batterij)\n')
kop = f'{"Maand":<9} {"Piek kW":>8} {"Vast €":>9} {"Dyn €":>9} {"Verschil €":>12} {"Verschil %":>11}'
print(kop)
print('-' * 62)
for m, r in maand_a.iterrows():
    print(f'{str(m):<9} {r["piek_kw"]:>8.2f} {r["totaal_vast"]:>9.2f}'
          f' {r["totaal_dyn"]:>9.2f} {r["verschil"]:>+12.2f} {r["verschil_pct"]:>+10.1f}%')
print('-' * 62)
print(f'{"TOTAAL":<9} {"":>8} {maand_a["totaal_vast"].sum():>9.2f}'
      f' {maand_a["totaal_dyn"].sum():>9.2f}'
      f' {maand_a["verschil"].sum():>+12.2f}'
      f' {maand_a["verschil"].sum()/maand_a["totaal_vast"].sum()*100:>+10.1f}%')


MAANDOVERZICHT — SCENARIO A (zonder batterij)

Maand      Piek kW    Vast €     Dyn €   Verschil €  Verschil %
--------------------------------------------------------------
2024-11       5.23    131.49    127.15        -4.35       -3.3%
2024-12       5.94    175.18    165.89        -9.29       -5.3%
2025-01       6.21    183.37    180.82        -2.55       -1.4%
2025-02       8.32    166.60    177.65       +11.05       +6.6%
2025-03       7.54    150.26    144.24        -6.02       -4.0%
2025-04       7.11    121.22    112.74        -8.48       -7.0%
2025-05       4.87     94.24     85.66        -8.58       -9.1%
2025-06       4.56    113.17    106.08        -7.09       -6.3%
2025-07       5.06    141.51    127.98       -13.53       -9.6%
2025-08       4.99    135.87    116.96       -18.91      -13.9%
2025-09       4.49    133.08    106.41       -26.67      -20.0%
2025-10       6.64    187.43    154.21       -33.22      -17.7%
2025-11       6.42    211.93    181.19       -30.74      

In [9]:
from plotly.subplots import make_subplots

# 1. Staafdiagram maandelijks (zonder verschil-lijn)
fig_a = go.Figure()
maanden_str = [str(m) for m in maand_a.index]
fig_a.add_trace(go.Bar(name='Vast (DATS24)',    x=maanden_str, y=maand_a['totaal_vast'], marker_color='steelblue'))
fig_a.add_trace(go.Bar(name='Dynamisch (EPEX)', x=maanden_str, y=maand_a['totaal_dyn'],  marker_color='darkorange'))
fig_a.update_layout(
    title='Maandelijkse elektriciteitskost: vast vs dynamisch (zonder batterij)',
    yaxis_title='Kost incl. BTW (€/maand)', barmode='group',
    legend=dict(orientation='h', y=1.1), height=420,
)
fig_a.show()

# 2. Dagelijkse variabele kosten
btw_lev = 1 + DATS24['btw']
btw_net = 1 + NETKOSTEN['btw']
btw_dyn = 1 + DYNAMISCH['btw']

df['_kost_vast_kw']   = df['ek_vast_nb'] * btw_lev + df['nk_kwartier'] * btw_net
df['_kost_dyn_nb_kw'] = df['ek_dyn_nb']  * btw_dyn + df['nk_kwartier'] * btw_net

dagkosten_a = df.groupby('datum')[['_kost_vast_kw', '_kost_dyn_nb_kw']].sum()
dagkosten_a.columns = ['vast', 'dyn']
dagkosten_a['verschil'] = dagkosten_a['dyn'] - dagkosten_a['vast']
top10_a = (dagkosten_a
           .reindex(dagkosten_a['verschil'].abs().nlargest(10).index)
           .sort_values('verschil', ascending=False))

print('TOP 10 DAGEN \u2014 GROOTSTE VERSCHIL DYNAMISCH VS VAST (variabele kosten, incl. BTW)')
print(f'{"Datum":<12} {"Vast €":>8} {"Dyn €":>8} {"Verschil €":>12}')
print('-' * 46)
for d, r in top10_a.iterrows():
    print(f'{str(d):<12} {r["vast"]:>8.2f} {r["dyn"]:>8.2f} {r["verschil"]:>+12.2f}')

# 3. Per-dag plot: kosten (links) + dynamisch tarief (rechts)
titels = [t for d in top10_a.index for t in [str(d), '']]
fig_10a = make_subplots(
    rows=10, cols=2,
    subplot_titles=titels,
    vertical_spacing=0.035,
    horizontal_spacing=0.10,
    column_widths=[0.55, 0.45],
)
for idx, (dag, rij) in enumerate(top10_a.iterrows()):
    row = idx + 1
    sel = df[df['datum'] == dag].copy()
    tl  = [pd.Timestamp(sel['kwartier'].iloc[i]).strftime('%H:%M') for i in range(len(sel))]
    toon = (idx == 0)

    # Links: kosten per kwartier
    fig_10a.add_trace(go.Scatter(
        x=tl, y=sel['_kost_vast_kw'], name='Vast',
        mode='lines', line=dict(color='steelblue', width=1.5), showlegend=toon,
    ), row=row, col=1)
    fig_10a.add_trace(go.Scatter(
        x=tl, y=sel['_kost_dyn_nb_kw'], name='Dynamisch',
        mode='lines', line=dict(color='darkorange', width=1.5), showlegend=toon,
    ), row=row, col=1)

    # Rechts: dynamisch tarief per kwartier (EPEX + opslag)
    tarief_dyn = sel['epex_eur_mwh'].fillna(0) / 1000 + DYNAMISCH['opslag_kwh']
    tarief_vast = np.where(sel['tarief'] == 'dag', DATS24['dag_kwh'], DATS24['nacht_kwh'])
    kleuren_bar = ['crimson' if v < 0 else '#4a90d9' for v in tarief_dyn]
    fig_10a.add_trace(go.Bar(
        x=tl, y=tarief_dyn, name='Dyn tarief (€/kWh)',
        marker_color=kleuren_bar, opacity=0.75, showlegend=toon,
    ), row=row, col=2)
    fig_10a.add_trace(go.Scatter(
        x=tl, y=tarief_vast, name='Vast tarief (€/kWh)',
        mode='lines', line=dict(color='steelblue', width=1.2, dash='dash'), showlegend=toon,
    ), row=row, col=2)

fig_10a.update_layout(
    title='Top 10 dagen: kosten per kwartier & tariefsevolutie (Scenario A)',
    height=1800,
    legend=dict(orientation='h', y=1.015, font_size=11),
    barmode='relative',
)
for r in range(1, 11):
    fig_10a.update_yaxes(title_text='\u20ac/kwartier', row=r, col=1)
    fig_10a.update_yaxes(title_text='\u20ac/kWh',     row=r, col=2)
fig_10a.show()


TOP 10 DAGEN — GROOTSTE VERSCHIL DYNAMISCH VS VAST (variabele kosten, incl. BTW)
Datum          Vast €    Dyn €   Verschil €
----------------------------------------------
2024-12-11       4.91     7.58        +2.67
2026-04-05       4.75     2.30        -2.46
2025-03-30       3.90     1.44        -2.46
2025-08-31       6.18     3.66        -2.53
2025-10-03       7.98     5.33        -2.64
2025-12-10      11.00     7.75        -3.25
2025-04-13       5.74     2.38        -3.35
2024-12-22       6.22     2.79        -3.42
2025-10-05       6.37     2.85        -3.51
2025-10-26       8.29     3.50        -4.79


## 5. Scenario B — Batterij LP-optimalisatie

De stationaire thuisbatterij wordt per dag optimaal ingezet om de
energiekost onder het **dynamisch tarief** te minimaliseren.

**LP-formulering per dag (T = 96 kwartieren):**

```
Minimaliseer:  Σ_t [ import_t * koop_t - export_t * verkoop_t ]

Subject to:
  (1) SOC[t] = SOC[t-1] + charge[t]*η_c - discharge[t]/η_d
  (2) import[t] - export[t] = base_load[t] + charge[t] - discharge[t]
  (3) SOC_min ≤ SOC[t] ≤ SOC_max
  (4) 0 ≤ charge[t], discharge[t] ≤ P_max * Δt
  (5) import[t], export[t] ≥ 0
```

**Variabelen per dag:** charge(96) + discharge(96) + soc(96) + import(96) + export(96) = 480

De SOC aan het einde van elke dag wordt overgedragen naar de volgende dag.

In [11]:
def optimize_bat_dag(base_load: np.ndarray,
                     koop_prijs: np.ndarray,
                     verkoop_prijs: np.ndarray,
                     soc_start_kwh: float) -> dict:
    """
    LP-optimalisatie van batterijdispatch voor 1 dag (96 kwartieren).

    Parameters
    ----------
    base_load      : kWh per kwartier (positief = afname, negatief = injectie)
    koop_prijs     : €/kWh per kwartier (inkoopprijs stroom)
    verkoop_prijs  : €/kWh per kwartier (verkoopprijs bij teruglevering)
    soc_start_kwh  : SOC in kWh aan begin van dag

    Returns
    -------
    dict met charge, discharge, soc, grid_import, grid_export (allemaal kWh-arrays)
    en 'success' bool en 'soc_eind' float.
    """
    T   = len(base_load)
    dt  = 0.25  # uur per kwartier
    cap = BAT['cap_kwh']
    pmax_q  = BAT['max_kw'] * dt  # max kWh per kwartier
    soc_min = BAT['soc_min'] * cap
    soc_max = BAT['soc_max'] * cap
    ec = BAT['eta_c']
    ed = BAT['eta_d']

    # Variabelen: [charge(T), discharge(T), soc(T), import(T), export(T)]
    Nc, Nd, Ns, Ni, Ne = 0, T, 2*T, 3*T, 4*T
    n_vars = 5 * T

    # Doelfunctie: minimaliseer import*koop - export*verkoop
    c_obj = np.zeros(n_vars)
    c_obj[Ni:Ni+T] =  koop_prijs
    c_obj[Ne:Ne+T] = -verkoop_prijs

    # Gelijkheidsconstraints: A_eq * x = b_eq
    # (1) SOC dynamica: soc[t] - charge[t]*ec + discharge[t]/ed - soc[t-1] = soc_start if t==0
    # (2) Netverdeling: import[t] - export[t] - charge[t] + discharge[t] = base_load[t]
    A_eq = np.zeros((2*T, n_vars))
    b_eq = np.zeros(2*T)

    for t in range(T):
        # SOC-dynamica
        A_eq[t, Ns+t]  =  1.0
        A_eq[t, Nc+t]  = -ec
        A_eq[t, Nd+t]  =  1.0/ed
        if t > 0:
            A_eq[t, Ns+t-1] = -1.0
        b_eq[t] = soc_start_kwh if t == 0 else 0.0
        # Netverdeling
        A_eq[T+t, Ni+t] =  1.0
        A_eq[T+t, Ne+t] = -1.0
        A_eq[T+t, Nc+t] = -1.0
        A_eq[T+t, Nd+t] =  1.0
        b_eq[T+t] = base_load[t]

    bounds = (
        [(0, pmax_q)]  * T +   # charge
        [(0, pmax_q)]  * T +   # discharge
        [(soc_min, soc_max)] * T +  # soc
        [(0, None)] * T +      # import
        [(0, None)] * T        # export
    )

    res = linprog(c_obj, A_eq=A_eq, b_eq=b_eq, bounds=bounds, method='highs')

    if res.success:
        return {
            'success':     True,
            'charge':      res.x[Nc:Nc+T],
            'discharge':   res.x[Nd:Nd+T],
            'soc':         res.x[Ns:Ns+T],
            'grid_import': res.x[Ni:Ni+T],
            'grid_export': res.x[Ne:Ne+T],
            'soc_eind':    res.x[Ns+T-1],
            'cost':        res.fun,
        }
    return {'success': False, 'error': res.message,
            'charge': np.zeros(T), 'discharge': np.zeros(T),
            'soc': np.full(T, soc_start_kwh), 'grid_import': base_load.clip(0),
            'grid_export': (-base_load).clip(0), 'soc_eind': soc_start_kwh}

### 5a. Optimalisatie over alle dagen

De optimalisatie loopt dag per dag. De eindstand van de batterij aan het
einde van dag *d* is de beginstand van dag *d+1*.

In [12]:
# Koopprijzen en verkoopprijzen per kwartier voor dynamisch tarief
df['koop_dyn']   = df['epex_eur_mwh'].fillna(df['epex_eur_mwh'].median()) / 1000 + DYNAMISCH['opslag_kwh']
df['verkoop_dyn'] = df['epex_eur_mwh'].fillna(df['epex_eur_mwh'].median()) / 1000 - DYNAMISCH['injectie_korting']

# Arrays voor geoptimaliseerde flows
n = len(df)
charge_opt    = np.zeros(n)
discharge_opt = np.zeros(n)
soc_opt       = np.zeros(n)
import_opt    = np.zeros(n)
export_opt    = np.zeros(n)

# soc_start: beginstand batterij (fractie van cap_kwh).
# Staat in BAT-dict (cel 3). .get() als vangnet bij stale kernelstate.
soc_now = BAT.get('soc_start', 0.30) * BAT['cap_kwh']
datum_array = df['datum'].values
dagen = sorted(set(datum_array))

print(f'Optimaliseren over {len(dagen)} dagen...')
mislukt = 0
for dag in dagen:
    idx = np.where(datum_array == dag)[0]
    if len(idx) < 96:
        # Onvolledige dag: overslaan, geen batterijdispatch
        import_opt[idx]  = df['base_load_kwh'].iloc[idx].clip(lower=0).values
        export_opt[idx]  = (-df['base_load_kwh'].iloc[idx]).clip(lower=0).values
        soc_opt[idx]     = soc_now
        continue
    bl    = df['base_load_kwh'].iloc[idx].values
    koop  = df['koop_dyn'].iloc[idx].values
    verk  = df['verkoop_dyn'].iloc[idx].values
    res   = optimize_bat_dag(bl, koop, verk, soc_now)
    charge_opt[idx]    = res['charge']
    discharge_opt[idx] = res['discharge']
    soc_opt[idx]       = res['soc']
    import_opt[idx]    = res['grid_import']
    export_opt[idx]    = res['grid_export']
    soc_now = res['soc_eind']
    if not res['success']:
        mislukt += 1

df['import_opt']    = import_opt
df['export_opt']    = export_opt
df['charge_opt']    = charge_opt
df['discharge_opt'] = discharge_opt
df['soc_opt']       = soc_opt

print(f'Klaar. Mislukte optimalisaties: {mislukt}')
print(f'Totale afname optimaal : {import_opt.sum():.1f} kWh')
print(f'Totale injectie optimaal: {export_opt.sum():.1f} kWh')

Optimaliseren over 522 dagen...
Klaar. Mislukte optimalisaties: 0
Totale afname optimaal : 13644.4 kWh
Totale injectie optimaal: 1939.6 kWh


In [13]:
# Kosten scenario B: geoptimaliseerde batterij + dynamisch tarief
df['ek_dyn_opt'] = energiekosten_dynamisch(
    df['import_opt'], df['export_opt'], df['epex_eur_mwh'].fillna(df['epex_eur_mwh'].median())
)

# Variabele netkosten met geoptimaliseerde flows
# (capaciteitstarief berekend op basis van maandpiek import_opt)
df['nk_kwartier_opt'] = netkosten_kwartier(df.assign(
    afname_kwh=df['import_opt'], injectie_kwh=df['export_opt']
))

cap_opt = capaciteitstarief_maand(pd.Series(import_opt, index=df.index), df['kwartier'])

# Maandoverzicht alle scenario's
if not isinstance(cap_nb.index, pd.PeriodIndex):
    cap_nb.index  = cap_nb.index.to_period('M')
if not isinstance(cap_opt.index, pd.PeriodIndex):
    cap_opt.index = cap_opt.index.to_period('M')

maand_b = df.groupby('maand').agg(
    ek_dyn_opt  = ('ek_dyn_opt',      'sum'),
    nk_opt_var  = ('nk_kwartier_opt', 'sum'),
).round(2)
maand_b = maand_b.join(cap_opt[['piek_kw','cap_kost']].rename(
    columns={'piek_kw':'piek_opt','cap_kost':'cap_opt'}), how='left')
maand_b['cap_opt'] = maand_b['cap_opt'].fillna(0)

btw = 1 + NETKOSTEN['btw']
maand_b['totaal_dyn_opt'] = (
    (maand_b['ek_dyn_opt'] + DYNAMISCH['maandkost']) * (1 + DYNAMISCH['btw'])
    + (maand_b['nk_opt_var'] + maand_b['cap_opt'] + NETKOSTEN['meter_mnd'] + NETKOSTEN['heffingen_mnd']) * btw
)

# Combineer met scenario A
vergelijk = maand_a[['totaal_vast','totaal_dyn']].join(maand_b[['totaal_dyn_opt']])
vergelijk.columns = ['Vast (DATS24)', 'Dynamisch (geen bat)', 'Dynamisch (bat opt)']
vergelijk['Besparing bat'] = vergelijk['Dynamisch (geen bat)'] - vergelijk['Dynamisch (bat opt)']
vergelijk['Besparing dyn vs vast'] = vergelijk['Dynamisch (geen bat)'] - vergelijk['Vast (DATS24)']

print('\nMAANDVERGELIJKING (incl. netkosten + BTW)\n')
kop2 = f'{"Maand":<9} {"Vast":>8} {"Dyn":>8} {"Dyn+Bat":>9} {"Bat besparing":>14} {"Dyn-Vast":>10}'
print(kop2)
print('-' * 64)
for m, r in vergelijk.iterrows():
    print(f'{str(m):<9}'
          f' {r["Vast (DATS24)"]:>8.2f}'
          f' {r["Dynamisch (geen bat)"]:>8.2f}'
          f' {r["Dynamisch (bat opt)"]:>9.2f}'
          f' {r["Besparing bat"]:>+14.2f}'
          f' {r["Besparing dyn vs vast"]:>+10.2f}')
print('-' * 64)
print(f'{"TOTAAL":<9}'
      f' {vergelijk["Vast (DATS24)"].sum():>8.2f}'
      f' {vergelijk["Dynamisch (geen bat)"].sum():>8.2f}'
      f' {vergelijk["Dynamisch (bat opt)"].sum():>9.2f}'
      f' {vergelijk["Besparing bat"].sum():>+14.2f}'
      f' {vergelijk["Besparing dyn vs vast"].sum():>+10.2f}')


MAANDVERGELIJKING (incl. netkosten + BTW)

Maand         Vast      Dyn   Dyn+Bat  Bat besparing   Dyn-Vast
----------------------------------------------------------------
2024-11     131.49   127.15    122.33          +4.82      -4.35
2024-12     175.18   165.89    153.72         +12.18      -9.29
2025-01     183.37   180.82    180.16          +0.66      -2.55
2025-02     166.60   177.65    176.06          +1.59     +11.05
2025-03     150.26   144.24    127.36         +16.88      -6.02
2025-04     121.22   112.74     91.91         +20.83      -8.48
2025-05      94.24    85.66     69.51         +16.15      -8.58
2025-06     113.17   106.08     85.96         +20.13      -7.09
2025-07     141.51   127.98    115.75         +12.23     -13.53
2025-08     135.87   116.96    102.02         +14.94     -18.91
2025-09     133.08   106.41    100.00          +6.41     -26.67
2025-10     187.43   154.21    142.83         +11.38     -33.22
2025-11     211.93   181.19    179.24          +1.95     -3

In [14]:
# 1. Staafdiagram alle scenario's (zonder verschil-lijn)
fig_b = go.Figure()
mx = [str(m) for m in vergelijk.index]
for naam, kleur in [
    ('Vast (DATS24)',        'steelblue'),
    ('Dynamisch (geen bat)', 'darkorange'),
    ('Dynamisch (bat opt)',  'seagreen'),
]:
    fig_b.add_trace(go.Bar(name=naam, x=mx, y=vergelijk[naam], marker_color=kleur))
fig_b.update_layout(
    title="Maandelijkse elektriciteitskost: alle scenario's vergelijking",
    yaxis_title='Kost incl. BTW (€/maand)', barmode='group',
    legend=dict(orientation='h', y=1.1), height=420,
)
fig_b.show()

# 2. Dagelijkse variabele kosten scenario B
df['_kost_dyn_opt_kw'] = df['ek_dyn_opt'] * btw_dyn + df['nk_kwartier_opt'] * btw_net

dagkosten_b = df.groupby('datum')[
    ['_kost_vast_kw', '_kost_dyn_nb_kw', '_kost_dyn_opt_kw']
].sum()
dagkosten_b.columns = ['vast', 'dyn_nb', 'dyn_opt']
dagkosten_b['verschil_opt_vast'] = dagkosten_b['dyn_opt'] - dagkosten_b['vast']

top10_b = (dagkosten_b
           .reindex(dagkosten_b['verschil_opt_vast'].abs().nlargest(10).index)
           .sort_values('verschil_opt_vast', ascending=False))

print('TOP 10 DAGEN \u2014 GROOTSTE VERSCHIL DYN+BAT VS VAST (variabele kosten, incl. BTW)')
print(f'{"Datum":<12} {"Vast €":>8} {"Dyn €":>8} {"Bat+Dyn €":>11} {"Versch €":>10}')
print('-' * 54)
for d, r in top10_b.iterrows():
    print(f'{str(d):<12} {r["vast"]:>8.2f} {r["dyn_nb"]:>8.2f}'
          f' {r["dyn_opt"]:>11.2f} {r["verschil_opt_vast"]:>+10.2f}')

# 3. Per-dag plot: kosten (links) + dynamisch tarief (rechts)
titels_b = [t for d in top10_b.index for t in [str(d), '']]
fig_10b = make_subplots(
    rows=10, cols=2,
    subplot_titles=titels_b,
    vertical_spacing=0.035,
    horizontal_spacing=0.10,
    column_widths=[0.55, 0.45],
)
for idx, (dag, rij) in enumerate(top10_b.iterrows()):
    row = idx + 1
    sel = df[df['datum'] == dag].copy()
    tl  = [pd.Timestamp(sel['kwartier'].iloc[i]).strftime('%H:%M') for i in range(len(sel))]
    toon = (idx == 0)

    # Links: drie kostenlijnen
    for naam, kol, kleur in [
        ('Vast',        '_kost_vast_kw',    'steelblue'),
        ('Dyn',         '_kost_dyn_nb_kw',  'darkorange'),
        ('Bat+Dyn',     '_kost_dyn_opt_kw', 'seagreen'),
    ]:
        fig_10b.add_trace(go.Scatter(
            x=tl, y=sel[kol], name=naam,
            mode='lines', line=dict(color=kleur, width=1.5), showlegend=toon,
        ), row=row, col=1)

    # Rechts: tariefsevolutie
    tarief_dyn  = sel['epex_eur_mwh'].fillna(0) / 1000 + DYNAMISCH['opslag_kwh']
    tarief_vast = np.where(sel['tarief'] == 'dag', DATS24['dag_kwh'], DATS24['nacht_kwh'])
    kleuren_bar = ['crimson' if v < 0 else '#4a90d9' for v in tarief_dyn]
    fig_10b.add_trace(go.Bar(
        x=tl, y=tarief_dyn, name='Dyn tarief (€/kWh)',
        marker_color=kleuren_bar, opacity=0.75, showlegend=toon,
    ), row=row, col=2)
    fig_10b.add_trace(go.Scatter(
        x=tl, y=tarief_vast, name='Vast tarief (€/kWh)',
        mode='lines', line=dict(color='steelblue', width=1.2, dash='dash'), showlegend=toon,
    ), row=row, col=2)

fig_10b.update_layout(
    title="Top 10 dagen: kosten per kwartier & tariefsevolutie (Scenario B)",
    height=1800,
    legend=dict(orientation='h', y=1.015, font_size=11),
    barmode='relative',
)
for r in range(1, 11):
    fig_10b.update_yaxes(title_text='\u20ac/kwartier', row=r, col=1)
    fig_10b.update_yaxes(title_text='\u20ac/kWh',     row=r, col=2)
fig_10b.show()


TOP 10 DAGEN — GROOTSTE VERSCHIL DYN+BAT VS VAST (variabele kosten, incl. BTW)
Datum          Vast €    Dyn €   Bat+Dyn €   Versch €
------------------------------------------------------
2025-10-03       7.98     5.33        4.89      -3.09
2025-08-31       6.18     3.66        3.07      -3.11
2026-04-05       4.75     2.30        1.32      -3.44
2025-04-27       8.73     7.58        5.20      -3.53
2024-12-22       6.22     2.79        2.54      -3.68
2025-12-10      11.00     7.75        7.30      -3.70
2025-10-05       6.37     2.85        2.60      -3.76
2025-05-11       2.37     2.41       -1.53      -3.90
2025-04-13       5.74     2.38        1.82      -3.92
2025-10-26       8.29     3.50        3.41      -4.88


## Waarom lijkt het dynamisch tarief altijd goedkoper?

De analyse toont dat het dynamisch tarief in de meeste periodes goedkoper uitkomt
dan het vaste DATS24-tarief. Dat lijkt misschien verrassend, maar heeft een aantal
structurele oorzaken.

### 1. Gemiddelde EPEX-prijs lag onder het vaste tarief in de meetperiode

De DATS24-dagprijs is vastgesteld op basis van een langjarig gemiddelde plus
winstmarge. In periodes met veel hernieuwbare productie (zonnige of winderige
middagen) daalt de EPEX-groothandelsprijs sterk — soms tot nul of negatief.
Wie dynamisch afneemt, profiteert daar rechtstreeks van.

### 2. Negatieve EPEX-prijzen

Op de rechtergrafiek zijn de rode balkjes de kwartieren met een **negatieve
EPEX-prijs**: de energieleverancier betaalt jou om stroom te verbruiken.
Bij een vast tarief mis je dit volledig; bij een dynamisch tarief verlaagt het
je rekening. Op die specifieke kwartieren is het verschil het grootst.

### 3. Batterijoptimalisatie versterkt het effect (Scenario B)

De LP-optimalisatie laadt de batterij bewust tijdens goedkope (of negatieve)
uren en ontlaadt tijdens dure uren. Dat verhoogt de besparing bovenop het
tariefsvoordeel.

### 4. Vaste maandkosten zijn niet meegeteld in deze analyse

De vergelijking toont enkel **variabele kosten** (energieprijs + variabele
netkosten). In de praktijk rekent een dynamisch contract een hogere vaste
maandkost aan dan een standaard vasteprijscontract. Die extra kost kan een
deel van de besparing opeten — zeker in periodes met laag verbruik.

### 5. Selectie-effect in de top-10

De getoonde dagen zijn juist geselecteerd op basis van het **grootste verschil**.
Over het volledige jaar zijn er ook dagen waarop het dynamisch tarief duurder
uitvalt (hoge EPEX-piek in de ochtend of avond). De top-10 geeft dus een
eenzijdig beeld; het totaaloverzicht per maand is een betere indicator.

### Conclusie

Het dynamisch tarief is structureel goedkoper zolang het **gemiddelde EPEX-niveau**
(inclusief leveringsopslag) onder de vaste kWh-prijs blijft. Bij stijgende
groothandelsprijzen (zoals in 2021-2022) kan dit snel omdraaien. De batterij
verzacht dat risico door goedkope uren te bufferen.


## 6. Samenvatting

De onderstaande cel berekent het totaaloverzicht over de volledige meetperiode
en extrapolert naar een jaarschatting.

**Methodologische beperkingen:**

- Het LP-model minimaliseert **energiekosten per dag**; het capaciteitstarief
  (maandpiek) is niet in de doelfunctie opgenomen. Voor volledig piekmijden:
  voeg een maandpiek-constraint toe aan het LP.
- EV-laden is als vaste belasting beschouwd (niet geoptimaliseerd).
- Solaire productie is impliciet verwerkt via reconstructie van de basisbelasting.
- DATS24-tariefwaarden zijn april-2026 schattingen; je eigen contract kan afwijken.
- Extrapolatie naar jaarcijfers veronderstelt dat de meetperiode representatief is.


In [15]:
import math

# -- Totalen over de volledige meetperiode --------------------------------
totaal_vast    = vergelijk['Vast (DATS24)'].sum()
totaal_dyn_nb  = vergelijk['Dynamisch (geen bat)'].sum()
totaal_dyn_opt = vergelijk['Dynamisch (bat opt)'].sum()

dyn_vs_vast    = totaal_dyn_nb  - totaal_vast
opt_vs_vast    = totaal_dyn_opt - totaal_vast
bat_besparing  = totaal_dyn_nb  - totaal_dyn_opt

# -- Periodelengte en jaarextrapolatie ------------------------------------
n_dagen   = df["datum"].nunique()
jaar_fac  = 365.25 / n_dagen

jaar_vast    = totaal_vast    * jaar_fac
jaar_dyn_nb  = totaal_dyn_nb  * jaar_fac
jaar_dyn_opt = totaal_dyn_opt * jaar_fac

# -- Opmaak ---------------------------------------------------------------
lijn = '-' * 70
print('SAMENVATTING TARIEFVERGELIJKING')
print('Variabele kosten (energiekosten + variabele netkosten, incl. BTW)')
print('Vaste maandkosten (meter, heffingen, levering) zijn hierin NIET inbegrepen')
print(lijn)
print('Meetperiode  :', df['datum'].min(), '->', df['datum'].max(),
      '(', n_dagen, 'dagen)')
print(lijn)

print()
print('TOTAAL OVER MEETPERIODE')
header = '  {:<30} {:>10}  {:>10}  {:>10}'.format(
    'Scenario', 'Kost (€)', 'vs Vast (€)', 'vs Vast (%)')
print(header)
print('  ' + '-' * 64)
rows_data = [
    ('Vast DATS24 (geen bat)',        totaal_vast,    0.0,           0.0),
    ('Dynamisch EPEX (geen bat)',     totaal_dyn_nb,  dyn_vs_vast,   dyn_vs_vast  / totaal_vast * 100),
    ('Dynamisch EPEX + bat opt',      totaal_dyn_opt, opt_vs_vast,   opt_vs_vast  / totaal_vast * 100),
]
for naam, kost, diff, pct in rows_data:
    diff_str = '         —' if diff == 0 else '{:>+10.2f}'.format(diff)
    pct_str  = '         —' if diff == 0 else '{:>+9.1f}%'.format(pct)
    print('  {:<30} {:>10.2f}  {}  {}'.format(naam, kost, diff_str, pct_str))
print('  ' + '-' * 64)
print('  {:<30} {:>+10.2f}'.format('  Batterijwinst (dyn→bat opt)', -bat_besparing))

print()
print('JAARSCHATTING (extrapolatie op basis van ' + str(n_dagen) + ' meetdagen)')
print(header)
print('  ' + '-' * 64)
rows_jaar = [
    ('Vast DATS24 (geen bat)',        jaar_vast,    0.0,                              0.0),
    ('Dynamisch EPEX (geen bat)',     jaar_dyn_nb,  jaar_dyn_nb  - jaar_vast,         (jaar_dyn_nb  - jaar_vast) / jaar_vast * 100),
    ('Dynamisch EPEX + bat opt',      jaar_dyn_opt, jaar_dyn_opt - jaar_vast,         (jaar_dyn_opt - jaar_vast) / jaar_vast * 100),
]
for naam, kost, diff, pct in rows_jaar:
    diff_str = '         —' if diff == 0 else '{:>+10.2f}'.format(diff)
    pct_str  = '         —' if diff == 0 else '{:>+9.1f}%'.format(pct)
    print('  {:<30} {:>10.2f}  {}  {}'.format(naam, kost, diff_str, pct_str))
print('  ' + '-' * 64)
print('  {:<30} {:>+10.2f}'.format(
    '  Batterijwinst/jaar', -(jaar_dyn_nb - jaar_dyn_opt)))

print()
print('TERUGVERDIENTIJD BATTERIJ (schatting)')
bat_jaar = jaar_dyn_nb - jaar_dyn_opt
if bat_jaar > 0:
    for prijs in [3000, 4000, 5000, 6000]:
        tvt = prijs / bat_jaar
        print('  Investeringskost {:>5} €  ->  {:5.1f} jaar terugverdientijd'.format(prijs, tvt))
else:
    print('  Batterij levert geen besparing in dit scenario.')


SAMENVATTING TARIEFVERGELIJKING
Variabele kosten (energiekosten + variabele netkosten, incl. BTW)
Vaste maandkosten (meter, heffingen, levering) zijn hierin NIET inbegrepen
----------------------------------------------------------------------
Meetperiode  : 2024-11-01 -> 2026-04-06 ( 522 dagen)
----------------------------------------------------------------------

TOTAAL OVER MEETPERIODE
  Scenario                         Kost (€)  vs Vast (€)  vs Vast (%)
  ----------------------------------------------------------------
  Vast DATS24 (geen bat)            2797.34           —           —
  Dynamisch EPEX (geen bat)         2556.84     -240.50       -8.6%
  Dynamisch EPEX + bat opt          2384.65     -412.70      -14.8%
  ----------------------------------------------------------------
    Batterijwinst (dyn→bat opt)     -172.19

JAARSCHATTING (extrapolatie op basis van 522 meetdagen)
  Scenario                         Kost (€)  vs Vast (€)  vs Vast (%)
  --------------------------